# Human-in-the-Loop: প্রাক-কর্ম গেট, ঝুঁকি স্তরীকরণ, এবং অডিট লগিং

এই পাঠের README পরিচিতি দেয় Human-in-the-Loop এর সাথে একটি সংক্ষিপ্ত অংশ যা ব্যবহারকারীকে `APPROVE` বা `REJECT` চাওয়া হয় যখন এজেন্ট ইতিমধ্যে একটি প্রতিক্রিয়া তৈরি করেছে। সেই প্যাটার্নটি একটি ভালো শুরু, তবে প্রোডাকশন HITL বাস্তবায়নে সাধারণত আরও তিনটি উপাদানের দরকার হয়:

1. একটি **প্রাক-কর্ম গেট** যা এজেন্ট যেকোন ঝুঁকিপূর্ণ ধাপ সম্পাদনের **আগে** চলে, যাতে খরচ, অপরিবর্তনীয়তা এবং বিলম্ব নিয়ন্ত্রণে থাকে।
2. **ঝুঁকি স্তরীকরণ**, যাতে কম ঝুঁকির কাজ স্বয়ংক্রিয়ভাবে সম্পন্ন হয়, মধ্যম ঝুঁকির কাজ ব্যাচ-অনুমোদিত হয়, এবং শুধু উচ্চ ঝুঁকির কাজেই একজন মানুষের অনুমোদনের প্রয়োজন হয়।
3. একটি **অডিট লগ এবং সংশোধন লুপ**, যাতে প্রতিটি গেট সিদ্ধান্ত JSONL হিসেবে রেকর্ড হয়, এবং প্রত্যাখ্যান এজেন্টকে একটি কাঠামোবদ্ধ কারণ সহ পুনরায় প্রম্পট করে শুধুমাত্র `Revising...` প্রিন্ট করার পরিবর্তে।

এই নোটবুকটি একই মৌলগুলোর উপর `06-system-message-framework.ipynb` এর উপর ভিত্তি করে প্রতিটি উপাদান তৈরি করে। এটি `DEMO_MODE = True` তে সম্পূর্ণ রকম রান করে (অন্তর্বর্তী ইনপুটের প্রয়োজন নেই) অথবা প্রকৃত `input()` প্রম্পটের সাথে যখন `DEMO_MODE = False` হয়। নোট করুন: DEMO_MODE তে তৃতীয় লক্ষ্যটির পুনরায় চেষ্টা স্ক্রিপ্ট করা হয়েছে তাই লুপের প্রক্রিয়া শেষ পর্যন্ত দৃশ্যমান। প্রকৃত সংশোধন-চালিত পুনর্বর্গীকরণ `DEMO_MODE = False` এবং একজন অপারেটরের প্রয়োজন।

**পরিধির বাইরে (অন্যান্য পাঠে সমাধান করা হয়েছে):** প্রমাণীকরণ এবং প্রবেশাধিকার নিয়ন্ত্রণ (Lesson 06 README threat 2), টুল-কল মিডলওয়্যার (Lesson 14 MAF গভীর আলোচনা), মাল্টি-এজেন্ট বিতর্ক প্যাটার্ন।

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## প্যাটার্ন ১: প্রি-অ্যাকশন গেট

README এর HITL স্নিপেট প্রথমে এজেন্টকে কল করে, তারপর ব্যবহারকারীর কাছে আউটপুট অনুমোদনের জন্য অনুরোধ করে। এটি একটি **পোস্ট-অ্যাকশন** ফ্লো। এজেন্ট ইতিমধ্যে কার্যকর হয়েছে, তাই LLM কল খরচ ইতিমধ্যে পরিশোধিত হয়েছে, এবং যেকোনো সাইড ইফেক্ট (ইমেইল পাঠানো, ডাটাবেসের সারি লেখা, মন্তব্য পোস্ট করা) ইতিমধ্যে ঘটেছে।

একটি **প্রি-অ্যাকশন** ফ্লো ঝুঁকিপূর্ণ ধাপের আগে গেট বসায়। এজেন্ট কর্ম প্রস্তাব করে, গেট সিদ্ধান্ত নেয় এটি কার্যকর করতে হবে কিনা, এবং অনুমোদনের পরই সাইড ইফেক্ট ঘটে।

| দিক | পোস্ট-অ্যাকশন অনুমোদন (README স্নিপেট) | প্রি-অ্যাকশন গেট (এই নোটবুক) |
|---|---|---|
| কখন অনুমোদন করা হয়? | এজেন্ট আউটপুট উত্পাদনের পরে | কোনো সাইড-ইফেক্ট কার্যকর হওয়ার আগে |
| প্রত্যাখ্যানের সময় LLM খরচ | ইতিমধ্যেই পরিশোধিত | শুধুমাত্র প্রস্তাবনার জন্য পরিশোধিত, ক্রিয়ার জন্য নয় |
| অপরিবর্তনীয় সাইড ইফেক্ট | সম্ভব (কর্ম ইতিমধ্যেই হয়েছে) | রোধ করা হয়েছে |
| নিরীক্ষণের স্পষ্টতা | অনুমোদন একটি প্রিন্ট স্টেটমেন্ট | অনুমোদন একটি টাইমস্ট্যাম্প, কর্ম, কারণ সহ JSON রেকর্ড |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## প্যাটার্ন ২: ঝুঁকি স্তরীকরণ

প্রত্যেকটি ক্রিয়াকলাপের জন্য মানব অনুমোদনের প্রয়োজন নেই। একটি পাবলিক API-এর বিরুদ্ধে শুধুমাত্র পড়ার জন্য একটি অনুসন্ধান গ্রাহক ইমেল পাঠানোর থেকে আলাদা। উভয়কেই একইভাবে বিবেচনা করলে অপারেটরের মনোযোগ নষ্ট হয় এবং এজেন্ট ধীরগতি হয়।

একটি সহজ ৩-স্তরের মডেল:

| স্তর | উদাহরণ | অনুমোদনের প্রবাহ |
|---|---|---|
| `নিম্ন` (শুধুমাত্র পড়া) | একটি জ্ঞানভাণ্ডার অনুসন্ধান, ফ্লাইট অপশন খোঁজা, একটি পাবলিক ওয়েব পেজ আনার কাজ | স্বয়ংক্রিয়ভাবে সম্পাদন, নিরীক্ষার জন্য লগ করা হয় |
| `মধ্য` (সস্তা পরিবর্তন) | একটি ফলাফল ক্যাশ করা, কাউন্টার বাড়ানো, একটি রিমাইন্ডার নির্ধারণ | স্বয়ংক্রিয়ভাবে সম্পাদন, তবে দৈনিক পর্যালোচনার জন্য গুচ্ছবদ্ধ |
| `উচ্চ` (বাহ্যিক বা অপরিবর্তনীয়) | একটি ইমেল পাঠানো, একটি কার্ড থেকে টাকা চার্জ করা, একটি পাবলিক চ্যানেলে পোস্ট করা | মানুষের অনুমোদনের জন্য ব্লক করা |

এটি একটি স্তরীকরণ। প্রোডাকশন সিস্টেমগুলি প্রায়ই আরও সূক্ষ্ম স্তর ব্যবহার করে (যেমন, AWS IAM অনুমতি স্তর, ভূমিকা-ভিত্তিক অ্যাক্সেস স্তর)। নীচের ৩-স্তর সংস্করণটি এমন একটি এজেন্টের জন্য সবচেয়ে ছোট উপযুক্ত সংস্করণ যা শুধুমাত্র পড়া এবং প্রভাব ফেলা ক্রিয়াকলাপ একসঙ্গে মিশ্রিত করে।

নীচের শ্রেণীবিভাজক কীওয়ার্ড হিউরিস্টিক ব্যবহার করে যাতে ডেমোটি নির্ধারিত এবং সস্তা থাকে। একটি প্রোডাকশন সিস্টেমে আপনি এটি একটি শেখানো শ্রেণীবিভাজক বা একটি নীতিমালা ইঞ্জিন দিয়ে প্রতিস্থাপন করবেন।

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## প্যাটার্ন ৩: অডিট লগ এবং সংশোধনের লুপ

`print("Response approved.")` একটি অডিট লগ নয়। বিশ্বাসের জন্য, প্রতিটি গেট সিদ্ধান্ত একটি স্ট্রাকচার্ড ইভেন্ট হিসেবে রেকর্ড করা উচিত যা আপনি পরে কোয়েরি করতে, পুনরায় চালাতে বা একটি ঘটনা পর্যালোচনার সাথে সংযুক্ত করতে পারেন।

দুইটি অংশ:

১. **শুধুমাত্র যোগ করার মতো JSONL।** প্রতি সিদ্ধান্তের জন্য একটি লাইন, যার মধ্যে থাকে টাইমস্ট্যাম্প, অ্যাকশন, স্তর, সিদ্ধান্ত, কারণ। সহজে grep করা যায়, পরে একটি বাস্তব লগ স্টোরে প্রেরণ করাও সহজ।  
২. **প্রত্যাখ্যানের উপর সংশোধনের লুপ।** যখন গেট `deny` রিটার্ন করে, এজেন্ট পুনরায় নিজেকে প্রত্যাখ্যানের কারণটি প্রসঙ্গ সহ প্রম্পট দেয়, যাতে পরবর্তী প্রস্তাব সমস্যাটি এড়াতে পারে।

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## অতিরিক্ত সরঞ্জামসমূহ

কয়েকটি অন্যান্য পাবলিক প্রকল্প এই HITL প্যাটার্নগুলির বিভিন্ন রূপ বাস্তবায়ন করে। আপনার স্ট্যাকের সাথে মানানসই কী তা খুঁজে পেতে পদ্ধতিগুলি তুলনা করুন:

- **LangChain** human-in-the-loop টুল র‍্যাপার্স ([ডক্স](https://python.langchain.com/docs/integrations/tools/human_tools)): ড্রপ-ইন টুল র‍্যাপার্স যা মানব ইনপুটের জন্য কার্যকলাপ থামে।
- **AutoGen** `UserProxyAgent` ([v0.2 ডক্স](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ এটি পুনর্গঠন করেছে): বহু-এজেন্ট কথোপকথনে মানবকে প্রতিনিধিত্ব করার জন্য একটি এজেন্ট রোল ব্যবহার করে।
- **Semantic Kernel** ফাংশন ফিল্টারসমূহ ([ডক্স](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): প্রতিটি ফাংশন কলের চারপাশে চলমান মিডলওয়্যার, যা গেটিং লজিকের জন্য উপযুক্ত।

প্রত্যেক প্রকল্প এই তিনটি সাব-প্যাটার্ন আলাদা ভাবে পরিচালনা করে: LangChain এগুলোকে টুল হিসেবে বেঁধে দেয়, AutoGen এজেন্ট রোল ব্যবহার করে, Semantic Kernel মিডলওয়্যার ফিল্টার ব্যবহার করে। আপনার নিজস্ব এজেন্টের ডিজাইন বেছে নেওয়ার আগে একটি বা দুইটি ইমপ্লিমেন্টেশন সম্পূর্ণ পড়ুন।

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
